In [2]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
# import warnings
# warnings.filterwarnings("ignore")
import sys
sys.path.insert(0, '/home/kat/Repos/SALSA/')

import os
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
%%time

ddir = '/home/kat/Repos/SALSA/data/chembl/'
df = pd.read_csv(os.path.join(ddir,'chembl_augmented_valid.csv'),usecols=['SMILES'])
df.columns = ['smiles']
display(df)

,smiles
0,CCO
1,OCC
2,C
3,CO
4,OC
...,...
6267871,O(C1C(O)C(n2c(=O)nc(N)cc2)OC1COP(O)(OC1C(O)C(n...
6267872,CC1=CN(C2CC(OP(O)(=O)OCC3OC(C(O)C3OP(O)(=O)OCC...
6267873,C1(O)C(OP(O)(=O)OCC2C(OP(=O)(O)OCC3OC(n4c5[nH]...
6267874,n1(C2CC(OP(O)(=O)OCC3OC(n4c(=O)nc(N)cc4)C(O)C3...


CPU times: user 4.08 s, sys: 440 ms, total: 4.52 s
Wall time: 4.52 s


In [7]:
import multiprocessing
max_threads = multiprocessing.cpu_count()

print(max_threads)

28


In [8]:
%%time
from utilities.rdkit_utils import get_cansmiles
from pandarallel import pandarallel

pandarallel.initialize(use_memory_fs=False)

df['cansmiles'] = df.smiles.parallel_apply(get_cansmiles)

INFO: Pandarallel will run on 14 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


[17:15:50] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 9 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 64 65 66 67 69 73 74 75 76 79 80 81 82
[17:15:51] Can't kekulize mol.  Unkekulized atoms: 0 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 33 34 35 36 37 38 39 40 41 42 68 69 70 71 72


CPU times: user 2.78 s, sys: 1.35 s, total: 4.13 s
Wall time: 4min 43s


In [28]:
display(df)

## Drop nans.
df = df.dropna()

## Drop empty strings.
df = df[df['cansmiles'].str.len() >= 1]

## De-duplicate. 
df_can = df.drop_duplicates(subset='cansmiles',keep='first')[['cansmiles']]
df_can.columns = ['Smiles']
display(df_can)

## Save csv.
fname = '/home/kat/Repos/SALSA/SALSA_cleanup/data/chembl/chembl_valid.csv'
df_can.to_csv(fname,index=False)

,smiles,cansmiles
0,CCO,CCO
1,OCC,CCO
2,C,C
3,CO,CO
4,OC,CO
...,...,...
6267871,O(C1C(O)C(n2c(=O)nc(N)cc2)OC1COP(O)(OC1C(O)C(n...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267872,CC1=CN(C2CC(OP(O)(=O)OCC3OC(C(O)C3OP(O)(=O)OCC...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267873,C1(O)C(OP(O)(=O)OCC2C(OP(=O)(O)OCC3OC(n4c5[nH]...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267874,n1(C2CC(OP(O)(=O)OCC3OC(n4c(=O)nc(N)cc4)C(O)C3...,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...


,Smiles
0,CCO
2,C
3,CO
5,NCCS
8,NCCN
...,...
6267852,Cc1cn(C2CN(P(=O)(OCC3CN(P(=O)(OCC4CNCC(n5cnc6c...
6267856,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267864,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...
6267868,C=C1NC(=O)C(C)=CN1C1CC(OP(=O)(O)OCC2OC(n3cnc4c...


In [29]:
## Save only SMILES that are <= 120 chars.
df_lte120 = df_can[df_can['Smiles'].str.len() <= 120]
## O
df_lte120 = df_can[df_can['Smiles'].str.len() <= 120]
display(df_lte120)

fname = '/home/kat/Repos/SALSA/SALSA_cleanup/data/chembl/chembl_valid_lte120.csv'
df_lte120.to_csv(fname,index=False)

,Smiles
0,CCO
2,C
3,CO
5,NCCS
8,NCCN
...,...
6179874,CC1C2CCC3C4CCC5Cc6nc7c(nc6CC5(C)C4CC(=O)C32COC...
6191186,CNC(=O)c1cc(Oc2ccc(Nc3ccc(Cl)c(C(F)(F)F)c3)cc2...
6204141,Cc1cc(C)c2c(c1)N(CCCN1CCN(c3ccccc3)CC1)C(=O)c1...
6205629,CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC(c5ccc(F)cc5F...


In [30]:
## Read back as test.
t = pd.read_csv(fname)
t

,Smiles
0,CCO
1,C
2,CO
3,NCCS
4,NCCN
...,...
1476101,CC1C2CCC3C4CCC5Cc6nc7c(nc6CC5(C)C4CC(=O)C32COC...
1476102,CNC(=O)c1cc(Oc2ccc(Nc3ccc(Cl)c(C(F)(F)F)c3)cc2...
1476103,Cc1cc(C)c2c(c1)N(CCCN1CCN(c3ccccc3)CC1)C(=O)c1...
1476104,CCC(C)n1ncn(-c2ccc(N3CCN(c4ccc(OC(c5ccc(F)cc5F...
